In [1]:
import numpy as np
import pandas as pd
import h5py
import argparse
import matplotlib.pyplot as plt
from scipy.fft import rfft, rfftfreq
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import beta
import json
import sys
import os
import time
import glob
from matplotlib.lines import Line2D
import traceback
import yaml
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LogNorm  # BEGIN: Changed to LogNorm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from cmcrameri import cm  # BEGIN: Import bamako colormap
import imageio


In [2]:
def load_yaml(yaml_path):
    with open(yaml_path, "r") as f:
        return yaml.safe_load(f)


def get_sipm_rel_pos(lrs_geometry_yaml, adc, channel):
    """
    Reproduces the logic of Geometry.get_sipm_rel_pos().

    Returns:
        (tpc, side, vert_pos)
        or (-1, -1, -1) if adc/channel is unused
    """
    tpc = -1
    det = -1

    # Find which TPC/detector this adc+channel belongs to
    for tpc_temp, det_map in lrs_geometry_yaml["det_adc"].items():
        for det_temp, adc_map in det_map.items():
            if adc_map == adc:
                if channel in lrs_geometry_yaml["det_chan"][tpc_temp][det_temp]:
                    tpc = tpc_temp
                    det = det_temp

    if tpc == -1 or det == -1:
        return -1, -1, -1

    side = lrs_geometry_yaml["det_side"][det]
    vert_pos = lrs_geometry_yaml["ch_to_vert_bin"][adc][channel]

    return tpc, side, vert_pos


def get_sipm_abs_pos_array(lrs_geometry_yaml, tpc_offsets):
    """
    Build SiPM absolute positions as a NumPy array indexed by [adc, channel].

    Parameters
    ----------
    lrs_geometry_yaml : dict
        Parsed light-geometry YAML.
    tpc_offsets : array-like, shape (4, 3)
        Detector TPC offsets, corresponding to det_geometry_yaml["tpc_offsets"]
        used in geometry.py.

    Returns
    -------
    sipm_abs_position : np.ndarray, shape (8, 48, 3)
        Absolute (x, y, z) position for each [adc, channel].
        Unused channels are filled with -1.
    """
    sipm_channels = ([4,5,6,7,8,9] + \
                     [10,11,12,13,14,15] + \
                     [20,21,22,23,24,25] + \
                     [26,27,28,29,30,31] + \
                     [36,37,38,39,40,41] + \
                     [42,43,44,45,46,47] + \
                     [52,53,54,55,56,57] + \
                     [58,59,60,61,62,63])

    tpc_offsets = np.asarray(tpc_offsets, dtype=float)

    sipm_abs_position = np.full((8, 64, 3), -1.0, dtype=float)

    n_sipm_centers = len(lrs_geometry_yaml["sipm_center"])
    half = n_sipm_centers // 2  # 24 for this file

    for adc in range(8):
        for channel in sipm_channels:
            #channel_id = sipm_channels[channel]
            tpc, side, vert_pos = get_sipm_rel_pos(lrs_geometry_yaml, adc, channel)

            if tpc == -1:
                continue

            tpc_channel = vert_pos + side * half
            sipm_center = np.asarray(lrs_geometry_yaml["sipm_center"][tpc_channel], dtype=float)
            tpc_center_offset = np.asarray(lrs_geometry_yaml["tpc_center_offset"][tpc], dtype=float)
            module_offset = tpc_offsets[tpc // 2]

            # X
            x_pos = module_offset[0] + tpc_center_offset[0]
            if tpc % 2 == 0:
                x_pos += sipm_center[0]
            else:
                x_pos -= sipm_center[0]

            # Y
            y_pos = module_offset[1] + tpc_center_offset[1] + sipm_center[1]

            # Z
            z_pos = module_offset[2] + tpc_center_offset[2] + sipm_center[2]

            sipm_abs_position[adc, channel] = [x_pos, y_pos, z_pos]

    return sipm_abs_position

In [3]:
def get_module_bounds(lrs_geometry_yaml, det_geometry_yaml):
    """
    Returns a list of 4 module bounds:
    [(xmin, xmax, ymin, ymax, zmin, zmax), ...]
    """
    tpc_offsets = np.asarray(det_geometry_yaml["tpc_offsets"], dtype=float)
    drift_length = float(det_geometry_yaml["drift_length"])

    # Infer y/z extent from light-geometry coordinates.
    # sipm_center gives the SiPM positions relative to each TPC face,
    # and tpc_center_offset shifts each TPC inside the module.
    tpc0_offset = np.asarray(lrs_geometry_yaml["tpc_center_offset"][0], dtype=float)
    sipm_centers = np.array(list(lrs_geometry_yaml["sipm_center"].values()), dtype=float)

    # Relative to module center
    y_rel = tpc0_offset[1] + sipm_centers[:, 1]
    z_rel = tpc0_offset[2] + sipm_centers[:, 2]

    ymin_rel, ymax_rel = np.min(y_rel), np.max(y_rel)
    zmin_rel, zmax_rel = np.min(z_rel), np.max(z_rel)

    bounds = []
    for module_center in tpc_offsets:
        cx, cy, cz = module_center

        xmin = (cx - drift_length)*1.05
        xmax = (cx + drift_length)*1.05
        ymin = (cy + ymin_rel)*1.05
        ymax = (cy + ymax_rel)*1.05
        zmin = (cz + zmin_rel)*1.05
        zmax = (cz + zmax_rel)*1.05

        bounds.append((xmin, xmax, ymin, ymax, zmin, zmax))

    return bounds

In [4]:
def draw_box_edges(ax, bounds, color="black", lw=1.2, ls="-"):
    xmin, xmax, ymin, ymax, zmin, zmax = bounds

    corners = np.array([
        [xmin, zmin, ymin],
        [xmax, zmin, ymin],
        [xmax, zmin, ymax],
        [xmin, zmin, ymax],
        [xmin, zmax, ymin],
        [xmax, zmax, ymin],
        [xmax, zmax, ymax],
        [xmin, zmax, ymax],
    ])

    edges = [
        (0, 1), (1, 2), (2, 3), (3, 0),  # bottom
        (4, 5), (5, 6), (6, 7), (7, 4),  # top
        (0, 4), (1, 5), (2, 6), (3, 7)   # verticals
    ]

    for i, j in edges:
        ax.plot(
            [corners[i, 0], corners[j, 0]],
            [corners[i, 1], corners[j, 1]],
            [corners[i, 2], corners[j, 2]],
            color=color,
            linewidth=lw,
            linestyle=ls
        )


In [5]:
def draw_cathode_plane(ax, bounds, x_cathode, color="dimgray", alpha=0.25, hatch=".."):
    _, _, ymin, ymax, zmin, zmax = bounds

    verts = [[
        [x_cathode, zmin, ymin],
        [x_cathode, zmin, ymax],
        [x_cathode, zmax, ymax],
        [x_cathode, zmax, ymin],
    ]]

    plane = Poly3DCollection(
        verts,
        facecolor=color,
        edgecolor=color,
        alpha=alpha,
        hatch=hatch,
        linewidths=0.8
    )
    ax.add_collection3d(plane)

In [6]:
SAMPLE_RATE = 0.016  # us per sample
SAMPLES = 600       # samples per waveform
adc14_max = 8191
adc14_16 = 2**2      # Conversion factor between 14 and 16 bit ADC

ADC_V_range = 2.0    # Voltage range of ADC
ADC_V_offset = -1.0  # Voltage offset


def get_noise_spectra(waveform, mask=None):
    # mask waveform, set to Nan
    if mask is not None:
        waveform = np.where(mask[..., np.newaxis], waveform, np.nan)
    # calculate the FFT of the waveform
    fft_N = rfft(waveform, axis=-1) / int(SAMPLES/2+1)
    # calculate the power spectrum
    power_spectra = 2*np.abs(fft_N)**2
    upper_quantile, power_spectrum, lower_quantile = np.nanquantile(power_spectra, [0.84, 0.5, 0.16], axis=0)
    # calculate the frequency bins
    freq_bins = rfftfreq(SAMPLES, d=SAMPLE_RATE * 1e-6)
    return freq_bins, power_spectra, power_spectrum, upper_quantile, lower_quantile
        # all events

# Get waveform information for a given event
def get_waveform_info(waveform, units='ADC16', mask=None, ths=None):
    if mask is not None:
        # mask is expected to be a list/array of event indices
        waveform = waveform[mask]
    if units == 'ADC14':
        waveform = waveform / adc14_16
    elif units == 'V':
        waveform = adc16_to_voltage(waveform)
    elif units != 'ADC16':
        raise ValueError("Units must be 'ADC14', 'ADC16', or 'V'.")
    # take the stdandard deviation of the first 50 samples
    noise = np.std(waveform[:, :, :, :50], axis=3)
    # take the mean of the first 50 sample
    baseline = np.mean(waveform[:, :, :, :50], axis=3)
    # subtract the baseline from the waveform
    max_value = np.max(waveform - baseline[:, :, :, np.newaxis], axis=3)
    
    wvfms = waveform[:, :, :, :]

    # return the baseline, max value, and clipped status
    return wvfms, noise, baseline, max_value
    
def adc16_to_voltage(adc_counts, mask=None):
    if mask is not None:
        adc_counts = np.where(mask[..., np.newaxis], adc_counts, 0)
    return adc_counts * (ADC_V_range + ADC_V_offset) / (adc14_max * adc14_16)

def get_max_value_mask(max_values, ptps, cs=None):
    # if ptps is a single value, convert it to a list of the same length as the number of ADCs
    if isinstance(ptps, (int, float)):
        ptps = [ptps] * 8
    elif len(ptps) != 8:
        raise ValueError("ptps must be a single value or a list of length 8")
    # Vectorized mask: True where max_values < ptps[adc] and channel_status == 0
    max_mask = max_values < (np.array(ptps)[np.newaxis, :, np.newaxis])
    if cs is not None:
        ch_mask = (cs == 0)[np.newaxis, :, :]
        mask = max_mask & ch_mask
    else:
        mask = max_mask
    return mask

In [7]:
channels = []
for group_start in range(0, 64, 16):
    channels.extend(range(group_start + 4, min(group_start + 16, 64)))
channel_status_csv = 'arcube_nearline/actions/light_dqm/channel_status.csv'
cs = None  # default if load fails

try:
    cs_df = pd.read_csv(channel_status_csv, header=None)
    cs = cs_df.to_numpy()
    print(f"Channel status loaded successfully from: {channel_status_csv}")
except FileNotFoundError:
    print(f"Channel status file not found, skipping: {channel_status_csv}")
except pd.errors.EmptyDataError:
    print(f"Channel status file is empty: {channel_status_csv}")
except Exception as e:
    print(f"Error loading channel status from {channel_status_csv}: {e}")

def plot_noise_spectra_channels_multi(
    freq_bins, spectra_dict, skip_bad_channels=True, nevts=None, 
    image_syntax="ColdComm", log_bool=True
):
    """
    Plot one subplot per (ADC, channel) pair.
    Each subplot can have multiple spectra curves (different runs/configs).

    Args:
        freq_bins: frequency bin edges
        spectra_dict: dict {label: noise_spectrum}, 
                      where noise_spectrum has shape (n_adcs, n_channels, n_freqs)
        skip_bad_channels: whether to skip bad channels from `cs`
        nevts: number of events processed (for title)
    """
    n_adcs = next(iter(spectra_dict.values())).shape[0]
    n_channels = next(iter(spectra_dict.values())).shape[1]


    # Mask the first frequency bin
    freq_mask = np.ones_like(freq_bins, dtype=bool)
    freq_mask[0] = False

    panel_idx = 0
    for i in range(n_adcs):          # ADC loop outer

        fig, axes = plt.subplots(len(channels), 1,
                                 figsize=(12, 2*len(channels))) #,
                                #sharex=True)
        extra = ''

        if len(channels) == 1:
            axes = [axes]

        for idx, ch in enumerate(channels):          # Channel loop inner
            if skip_bad_channels and cs is not None and cs.shape == (n_adcs, n_channels):
                if cs[i, ch] != 0:
                    extra = ', bad channel'
                else:
                    extra = ' '
            ax = axes[idx]
            #print(np.shape(spectra_dict.items()))
            
            for label, noise_spectrum in spectra_dict.items():
                if label=="Baseline":
                    previous_spectrum = noise_spectrum[i,ch][freq_mask]
                else:
                    new_spectrum = noise_spectrum[i,ch][freq_mask] #np.clip((noise_spectrum[i,ch][freq_mask] , a_min=0, a_max=None)
                    ax.step(
                        freq_bins[freq_mask]/1e6,
                        new_spectrum,
                        where="mid", label=label
                    )
                    previous_spectrum = noise_spectrum[i,ch][freq_mask]

            ax.set_title(f"ADC {i} Channel {ch} noise spectrum: {nevts} events{extra}")
            ax.set_ylabel("V^2 / bin")
            if log_bool == True:
                ax.set_yscale("log")
            ax.set_xlabel("Frequency (MHz)")
            #if crop: ax.set_ylim(5e-10, 1e-5)
            ax.set_xlim(0, 31.25)
            ax.grid(True, which="major", linestyle="-", linewidth=0.5)
            ax.legend(loc="upper right", fontsize="x-small")

        #axes[-1].set_xlabel("Frequency (MHz)")
        plt.tight_layout()
        
        #plt.savefig(f"{image_syntax}_ADC{i}_VGAscan.pdf")
        
    # # save as pdf
    # output_pdf = f"{args.tmp_dir}/{output_name}"
    # with PdfPages(output_pdf) as pdf:
    #     pdf.savefig()
    #     plt.close()



Channel status file not found, skipping: arcube_nearline/actions/light_dqm/channel_status.csv


In [8]:
files = {
    1091: "ADC Rack Only", 
    1090: "ADC Rack + R. Cables",
    1092: "ADC + VGA Rack", 
    1093: "ADC + VGA + MPOD", 
    1094: "ADC + VGA + MPOD + TTI", 
    1095: "SiPM HV: 48V", 
    1100: "SiPM HV: 1V, CRS Configured", 
    1102: "SiPM HV: 1V, CRS Triggering" 
}
nevents = 500 #number of events to take since there were some crashes
spectra_dict = {}

for ifile, label in files.items():
    test_bool =  (ifile<=1102 & ifile>=1090)
    if test_bool==1:
        filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run3/flowed_light/warm_commission/noisescan_20260709/mpd_run_data_rctl_*_00{ifile}_p00001.FLOW.hdf5'
    #if ifile==984:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/cosmics_bin15/mpd_run_data_rctl_{ifile}_p232.FLOW.hdf5'
    #if ifile<831:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/warm_commission/mpd_run_dbg_rctl_{ifile}.FLOW.hdf5'
    #else:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/postrun_status_check/TimerTrig_noise/mpd_run_data_rctl_{ifile}_p0.FLOW.hdf5'
    #elif ifile==934:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/source_rn_bin1/test_files/mpd_run_data_rctl_{ifile}_p40.FLOW.hdf5'
    #else:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/cosmics_hvramp_down_12162025/mpd_run_data_rctl_{ifile}_p4.FLOW.hdf5'
    #elif ifile==1395:
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/cold_commission/mpd_run_calib_rctl_{ifile}.FLOW.hdf5'
    #else:   
    #    filename = f'/global/cfs/cdirs/dune/www/data/2x2/nearline_run2/flowed_light/postrun_status_check/TimerTrig_noise/mpd_run_data_rctl_{ifile}_p0.FLOW.hdf5'
    file = h5py.File(filename, 'r')
    nevents_total = file["light/wvfm/data"]['samples'][:,:,:,:600].shape[0]
    if nevents_total < nevents:
        nevents = nevents_total
    sel_idx = np.linspace(0, nevents_total - 1, nevents, dtype=int)
    ptps = np.array([500.]*8)

    wvfms, noises, baselines, max_values = get_waveform_info(
        file["light/wvfm/data"]['samples'][:,:,:,:600], 'ADC16', mask=sel_idx, ths=ptps
    )
    print("No. of events used", len(wvfms))

    wvfms_v = adc16_to_voltage(wvfms[:nevents])
    freq_bins, noise_spectra, noise_spectrum, upper, lower = get_noise_spectra(
        wvfms_v, None
    )

    spectra_dict[label] = noise_spectrum

#plot_noise_spectra_channels_multi(
#    freq_bins, spectra_dict, skip_bad_channels=True, nevts=500, 
#    image_syntax="Coldcomm_noise_uncropped", log_bool=True
#)

#print(spectra_dict['Warm 1/15: 24dB, 50V'])


No. of events used 500
No. of events used 500


In [9]:
spectra_diffs_dict = {}
integral_diffs_dict = {}
diff_titles = ["10dB Cold, HV: 12/16 - 12/09"]

for item_idx in range(len(files.items())):
    if (item_idx%2)==0:
        title_idx = int(item_idx//2)
        key_list = [list(files.keys())[item_idx], list(files.keys())[item_idx+1]]

        noise_difference_array = spectra_dict[files[key_list[1]]] - spectra_dict[files[key_list[0]]]

        spectra_diffs_dict[diff_titles[title_idx]] = noise_difference_array
        integral_diffs_dict[diff_titles[title_idx]] = np.trapezoid(noise_difference_array[:,:,:45], axis=-1)

#plot_noise_spectra_channels_multi(
#    freq_bins, spectra_diffs_dict, skip_bad_channels=True, nevts=500, 
#    image_syntax="Coldcomm_noise_uncropped", log_bool=False
#)
print(np.shape(integral_diffs_dict["10dB Cold, HV: 12/16 - 12/09"]))

(8, 64)


In [10]:
# Example usage
lrs = load_yaml("/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/Commissioning_Code/2x2_LRS_OperatingScripts/LowLevel_Commissioning/light_module_desc-5.0.1.yaml")
geo = load_yaml("/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/Commissioning_Code/2x2_LRS_OperatingScripts/LowLevel_Commissioning/2x2.yaml")
# You must provide these from det_geometry_yaml["tpc_offsets"]
# Example placeholder:
tpc_offsets = np.array([
    [33.5, 0., 33.5],
    [33.5, 0., -33.5],
    [-33.5, 0., 33.5],
    [-33.5, 0., -33.5],
])

sipm_channels = ([4,5,6,7,8,9] + \
                    [10,11,12,13,14,15] + \
                    [20,21,22,23,24,25] + \
                    [26,27,28,29,30,31] + \
                    [36,37,38,39,40,41] + \
                    [42,43,44,45,46,47] + \
                    [52,53,54,55,56,57] + \
                    [58,59,60,61,62,63]
                    )

sipm_abs_position = get_sipm_abs_pos_array(lrs, tpc_offsets)

print(sipm_abs_position.shape)   # (8, 48, 3)
print(sipm_abs_position[3, 62])   # xyz of adc 0, channel 4

sipm_abs_position_flat = sipm_abs_position[:,sipm_channels,:].reshape(-1, 3)
print(np.shape(sipm_abs_position_flat))  # (384, 3)

(8, 64, 3)
[  3.06  54.85 -64.99]
(384, 3)


In [11]:
def plot_event_example(noise_arr, all_results, det_geometry_yaml, lrs_geometry_yaml, module=None, event=None, title=None, azimuths=40, elevation=20, output_path=None):

        # Normalize or set to zero if all zero
        
        noise_excess = np.clip(noise_arr, a_min=np.min(noise_arr[noise_arr>0]), a_max=None)

        # Set up colormap and normalization range
        vmin = np.min(noise_excess[noise_excess!=0])
        #print('MIN:', vmin)
        vmax = np.max(noise_excess)
        #print('MAX:', vmax)
        
        norm = LogNorm(1, vmax/vmin)  # END: Changed to LogNorm

        cmap = cm.navia_r # END: Use bamako colormap

        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')

        if module is not None:
            if module==13:
                channel_mod1 = np.arange(96, 192, 1)
                channel_mod3 = np.arange(288, 384, 1)
                channel_array = np.concatenate([channel_mod1, channel_mod3]).tolist()
                noise_excess = noise_excess[channel_array]
                all_results = all_results[channel_array]
            elif module==20:
                channel_mod0 = np.arange(0, 96, 1)
                channel_mod2 = np.arange(192, 288, 1)
                channel_array = np.concatenate([channel_mod0, channel_mod2]).tolist()
                noise_excess = noise_excess[channel_array]
                all_results = all_results[channel_array]
            else:
                channel_min = module * 96
                channel_max = (module + 1) * 96
                noise_excess = noise_excess[channel_min:channel_max]
                all_results = all_results[channel_min:channel_max]
        # Plot rectangles
        for result, value in zip(all_results, noise_excess/vmin):
            x, y, z_det, *_ = result
            if x > 60:
                x_offset = -29
            elif x > 0:
                x_offset = 29
            elif x < -60:
                x_offset = 29
            else:
                x_offset = -29
            corners = [
                [x, z_det, y-2],
                [x+x_offset, z_det, y-2],
                [x+x_offset, z_det, y+2],
                [x, z_det, y+2]
            ]
            color = cmap(norm(value))  # Map actual PDE % through normalized colormap
            rect = Poly3DCollection([corners], color=color, alpha=0.5)
            ax.add_collection3d(rect)

        module_bounds = get_module_bounds(lrs_geometry_yaml, det_geometry_yaml)

        for i, bounds in enumerate(module_bounds):
            draw_box_edges(ax, bounds, color="black", lw=1.3)
            x_cathode = det_geometry_yaml["tpc_offsets"][i][0]
            draw_cathode_plane(ax, bounds, x_cathode, color="gray", alpha=0.12, hatch="..")
        
        for i, bounds in enumerate(module_bounds):
            xmin, xmax, ymin, ymax, zmin, zmax = bounds

            # Bottom center in Y and Z
            y_text = ymin
            z_text = 0.5 * (zmin + zmax)

            # Decide outward direction
            x_center = 0.5 * (xmin + xmax)
            if x_center > 0:
                x_text = xmax - 15  # push outward on +X side
                ha = 'center'
            else:
                x_text = xmin + 15   # push outward on -X side
                ha = 'center'

            ax.text(
                x_text, z_text, y_text,
                f"Module {i}",
                fontsize=14,
                color="black",
                ha=ha,
                va='center'
            )

        #ax.plot(x, z, y, color='b', marker='o', markersize=2, ls='none', alpha=0.8, label='Muon track')
        # Colorbar setup — use the same normalization!
        sm = ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])  # Just to make matplotlib happy
        cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label(f'Relative Noise Increase')

        # Axes setup
        ax.view_init(elev=elevation, azim=azimuths)
        ax.set_xlabel('X (cm)')
        ax.set_ylabel('Z (cm)')
        ax.set_zlabel('Y (cm)')
        ax.set_xlim(-70, 70)
        ax.set_ylim(-70, 70)
        ax.set_zlim(-70, 70)
        ax.set_title(f'{title}', fontsize=16)

        #plt.savefig(os.path.join("pde_plots/", f'pde_{title}_{event}.png'))
        #plt.close()
        #plt.show()

        return fig
    


In [12]:
channel_status_csv = '/global/cfs/cdirs/dune/users/ajwhite/2x2_LRS_DataAssess/Commissioning_Code/2x2_LRS_OperatingScripts/LowLevel_Commissioning/channel_status.csv'
cs_df = pd.read_csv(channel_status_csv, header=None)
cs = cs_df.to_numpy()
#print(np.shape(cs))
#print(np.shape(integral_diffs_dict['24dB Warm, 50V: 1/15 - 9/02']))
noise_array = np.array(integral_diffs_dict["10dB Cold, HV: 12/16 - 12/09"])*(cs==0) #- np.array(integral_diffs_dict['24dB Warm, 50V: 1/15 - 9/02']))*(cs==0)#[:, sipm_channels]
#print(np.shape(noise_array))
noise_array_flat = noise_array[:, sipm_channels].reshape(-1)
print(np.shape(noise_array_flat))
#print(noise_array_flat)

(384,)


In [13]:
#plot_event_example(noise_arr=noise_array_flat, all_results=sipm_abs_position_flat, det_geometry_yaml=geo, lrs_geometry_yaml=lrs, module=3, title=f"Module 3 Noise Differences: {diff_titles[1]}")

In [14]:
n_frames=90
azimuths = np.linspace(0, 360, n_frames)

frames_dir = "./trigger_frames"
os.makedirs(frames_dir, exist_ok=True)
OUT=frames_dir

frame_files = []

for i, az in enumerate(azimuths):

    fig = plot_event_example(noise_arr=noise_array_flat, all_results=sipm_abs_position_flat, det_geometry_yaml=geo, lrs_geometry_yaml=lrs, module=None, title=f"All Modules Noise Differences: {diff_titles[0]}", azimuths=az, elevation=20, output_path=OUT)

    frame_path = os.path.join(frames_dir, f"frame_{i:04d}.png")
        
    fig.savefig(frame_path, dpi=100, bbox_inches="tight")
    plt.close(fig)

    frame_files.append(frame_path)

from PIL import Image

images = []
size = None

for f in frame_files:
    img = Image.open(f).convert("RGB")  # force same channel count
    if size is None:
        size = img.size                  # (width, height) from first frame
    img = img.resize(size)              # force same dimensions
    images.append(img)

imageio.mimsave(
    "./Noise_Cold_All_diff.gif",
    [imageio.core.asarray(img) for img in images],
    fps=7,
    loop=0
)

#images = [imageio.imread(f) for f in frame_files]
#imageio.mimsave("./Noise_diff.gif", images, fps=5, loop=0)
#os.rmdir(frames_dir)